In [1]:
import pandas as pd
import numpy as np

# -----------------------------
# Load datasets
# -----------------------------
train_df = pd.read_csv("train_datav3.csv")
test_df = pd.read_csv("test_datav3.csv")

# -----------------------------
# Columns to remove
# -----------------------------
cols_to_remove = [
    'cog_rad',
    'cog_sin',
    'cog_cos',
    'hdg_rad',
    'hdg_sin',
    'hdg_cos',
    'dlat',
    'dlon'
]

# Remove columns from both datasets
train_df = train_df.drop(columns=cols_to_remove)
test_df = test_df.drop(columns=cols_to_remove)

# -----------------------------
# Convert angles to radians
# -----------------------------
train_cog_rad = np.deg2rad(train_df["COG"])
train_heading_rad = np.deg2rad(train_df["HEADING"])

test_cog_rad = np.deg2rad(test_df["COG"])
test_heading_rad = np.deg2rad(test_df["HEADING"])

# -----------------------------
# Create new trig features
# -----------------------------
train_df["COG_cos"] = np.cos(train_cog_rad)
train_df["COG_sin"] = np.sin(train_cog_rad)

train_df["HEADING_cos"] = np.cos(train_heading_rad)
train_df["HEADING_sin"] = np.sin(train_heading_rad)


test_df["COG_cos"] = np.cos(test_cog_rad)
test_df["COG_sin"] = np.sin(test_cog_rad)

test_df["HEADING_cos"] = np.cos(test_heading_rad)
test_df["HEADING_sin"] = np.sin(test_heading_rad)

# -----------------------------
# Verify the results
# -----------------------------
print("TRAIN SHAPE:", train_df.shape)
print("TEST SHAPE:", test_df.shape)

print("\nTRAIN COLUMNS:")
print(train_df.columns.tolist())

print("\nTEST COLUMNS:")
print(test_df.columns.tolist())

print("\nTRAIN SAMPLE:")
print(train_df.head())

print("\nTEST SAMPLE:")
print(test_df.head())

seq_len = 12

KeyError: "['cog_rad', 'cog_sin', 'cog_cos', 'hdg_rad', 'hdg_sin', 'hdg_cos', 'dlat', 'dlon'] not found in axis"

In [2]:
import pandas as pd

# Convert timestamps so sorting works properly
train_df["TIME"] = pd.to_datetime(train_df["TIME"])
test_df["TIME"] = pd.to_datetime(test_df["TIME"])

# Sort by voyage and time
train_df = train_df.sort_values(["voyage_id", "TIME"]).reset_index(drop=True)
test_df = test_df.sort_values(["voyage_id", "TIME"]).reset_index(drop=True)

# Compute motion deltas within each voyage
train_df["dlat"] = train_df.groupby("voyage_id")["LAT"].shift(-1) - train_df["LAT"]
train_df["dlon"] = train_df.groupby("voyage_id")["LON"].shift(-1) - train_df["LON"]

test_df["dlat"] = test_df.groupby("voyage_id")["LAT"].shift(-1) - test_df["LAT"]
test_df["dlon"] = test_df.groupby("voyage_id")["LON"].shift(-1) - test_df["LON"]

# Remove rows where the target cannot exist (last ping of each voyage)
train_df = train_df.dropna(subset=["dlat", "dlon"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["dlat", "dlon"]).reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain sample:")
print(train_df.head())

print("\nTest sample:")
print(test_df.head())

Train shape: (27253, 16)
Test shape: (11740, 16)

Train sample:
        MMSI        LAT        LON               PERIOD  SPEED    COG  \
0  255806122  22.662888 -78.254898  2023-04-25 02:40:00   16.1  292.9   
1  255806122  22.690800 -78.330125  2023-04-25 02:55:00   15.9  291.2   
2  255806122  22.704480 -78.367100  2023-04-25 03:05:00   15.8  290.9   
3  255806122  22.715740 -78.397202  2023-04-25 03:10:00   15.8  293.0   
4  636017022  23.073705 -78.990980  2024-01-29 21:35:00   12.4  152.8   

   HEADING      voyage_id                TIME          dt   COG_cos   COG_sin  \
0    293.0  100_255806122 2023-04-25 02:40:00    840900.0  0.389124 -0.921185   
1    293.0  100_255806122 2023-04-25 02:55:00       900.0  0.361625 -0.932324   
2    292.0  100_255806122 2023-04-25 03:05:00       600.0  0.356738 -0.934204   
3    293.0  100_255806122 2023-04-25 03:10:00       300.0  0.390731 -0.920505   
4    152.0  100_636017022 2024-01-29 21:35:00  13660800.0 -0.889416  0.457098   

   HEADING

In [3]:
import numpy as np

def build_tcn_sequences(df, seq_len=4):

    feature_cols = [
        "LAT",
        "LON",
        "SPEED",
        "dt",
        "COG_cos",
        "COG_sin",
        "HEADING_cos",
        "HEADING_sin"
    ]

    target_cols = ["LAT", "LON"]

    X = []
    Y = []

    # group by voyage
    for voyage_id, group in df.groupby("voyage_id"):

        # ensure time ordering
        group = group.sort_values("TIME")

        features = group[feature_cols].values
        targets = group[target_cols].values

        # sliding window
        for i in range(len(group) - seq_len):

            X.append(features[i:i + seq_len])
            Y.append(targets[i + seq_len])

    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)

    return X, Y

In [4]:


X_train, Y_train = build_tcn_sequences(train_df, seq_len)
X_test, Y_test = build_tcn_sequences(test_df, seq_len)

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

print("X_test shape:", X_test.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (8717, 12, 8)
Y_train shape: (8717, 2)
X_test shape: (3862, 12, 8)
Y_test shape: (3862, 2)


In [5]:
print(X_train[0])
print(Y_train[0])

[[ 2.1866550e+01 -7.7097382e+01  1.6000000e+01  3.0677700e+07
   6.6783255e-01 -7.4431157e-01  6.5605903e-01 -7.5470960e-01]
 [ 2.1873720e+01 -7.7106094e+01  1.6000000e+01  3.0000000e+02
   6.6392618e-01 -7.4779809e-01  6.4278764e-01 -7.6604444e-01]
 [ 2.1892931e+01 -7.7129402e+01  1.6100000e+01  3.0000000e+02
   6.5474081e-01 -7.5585347e-01  6.4278764e-01 -7.6604444e-01]
 [ 2.1904535e+01 -7.7143456e+01  1.6000000e+01  3.0000000e+02
   6.6523033e-01 -7.4663818e-01  6.4278764e-01 -7.6604444e-01]
 [ 2.1919374e+01 -7.7161377e+01  1.6100000e+01  3.0000000e+02
   6.5342063e-01 -7.5699508e-01  6.4278764e-01 -7.6604444e-01]
 [ 2.1946260e+01 -7.7194061e+01  1.6000000e+01  3.0000000e+02
   6.6523033e-01 -7.4663818e-01  6.5605903e-01 -7.5470960e-01]
 [ 2.1949230e+01 -7.7197647e+01  1.6100000e+01  3.0000000e+02
   6.5342063e-01 -7.5699508e-01  6.5605903e-01 -7.5470960e-01]
 [ 2.1963640e+01 -7.7215248e+01  1.6000000e+01  3.0000000e+02
   6.5077424e-01 -7.5927132e-01  6.4278764e-01 -7.6604444e-01]


In [6]:
# reset dt for first ping of every voyage
train_df.loc[train_df.groupby("voyage_id").head(1).index, "dt"] = 0
test_df.loc[test_df.groupby("voyage_id").head(1).index, "dt"] = 0
train_df.groupby("voyage_id")["dt"].first().describe()

count    2721.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: dt, dtype: float64

In [7]:


X_train, Y_train = build_tcn_sequences(train_df, seq_len)
X_test, Y_test = build_tcn_sequences(test_df, seq_len)

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_test shape:", X_test.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (8717, 12, 8)
Y_train shape: (8717, 2)
X_test shape: (3862, 12, 8)
Y_test shape: (3862, 2)


In [8]:
train_df["dt"].describe()

count    27253.000000
mean       333.882508
std        194.307746
min          0.000000
25%        300.000000
50%        300.000000
75%        300.000000
max       1200.000000
Name: dt, dtype: float64

In [9]:
train_df["dlat"] = train_df.groupby("voyage_id")["LAT"].diff()
train_df["dlon"] = train_df.groupby("voyage_id")["LON"].diff()

test_df["dlat"] = test_df.groupby("voyage_id")["LAT"].diff()
test_df["dlon"] = test_df.groupby("voyage_id")["LON"].diff()

In [10]:
train_df["dlat"] = train_df["dlat"].fillna(0)
train_df["dlon"] = train_df["dlon"].fillna(0)

test_df["dlat"] = test_df["dlat"].fillna(0)
test_df["dlon"] = test_df["dlon"].fillna(0)

In [11]:
import numpy as np

def build_tcn_sequences(df, seq_len=4):

    feature_cols = [
        "LAT",
        "LON",
        "SPEED",
        "dt",
        "COG_cos",
        "COG_sin",
        "HEADING_cos",
        "HEADING_sin"
    ]

    target_cols = ["dlat", "dlon"]

    X = []
    Y = []

    for voyage_id, group in df.groupby("voyage_id"):

        group = group.sort_values("TIME")

        features = group[feature_cols].values
        targets = group[target_cols].values

        for i in range(len(group) - seq_len):

            X.append(features[i:i + seq_len])
            Y.append(targets[i + seq_len])

    X = np.array(X, dtype=np.float32)
    Y = np.array(Y, dtype=np.float32)

    return X, Y

In [12]:


X_train, Y_train = build_tcn_sequences(train_df, seq_len)
X_test, Y_test = build_tcn_sequences(test_df, seq_len)

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

print("X_test shape:", X_test.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (8717, 12, 8)
Y_train shape: (8717, 2)
X_test shape: (3862, 12, 8)
Y_test shape: (3862, 2)


In [13]:
Y_train

array([[ 0.01799 , -0.021778],
       [ 0.014185, -0.01715 ],
       [ 0.01183 , -0.01419 ],
       ...,
       [-0.00975 ,  0.025122],
       [-0.005627,  0.014167],
       [ 0.018346, -0.04822 ]], shape=(8717, 2), dtype=float32)

In [14]:
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_test_2d  = X_test.reshape(-1, X_test.shape[-1])

scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test_scaled  = scaler.transform(X_test_2d).reshape(X_test.shape)

X_train = X_train_scaled.astype(np.float32)
X_test  = X_test_scaled.astype(np.float32)

In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train)
Y_train_t = torch.tensor(Y_train)

X_test_t = torch.tensor(X_test)
Y_test_t = torch.tensor(Y_test)

train_dataset = TensorDataset(X_train_t, Y_train_t)
test_dataset = TensorDataset(X_test_t, Y_test_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [16]:
import torch.nn as nn

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.relu2 = nn.ReLU()

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) \
            if in_channels != out_channels else None

    def forward(self, x):

        out = self.conv1(x)
        out = self.relu1(out)

        out = self.conv2(out)
        out = self.relu2(out)

        res = x if self.downsample is None else self.downsample(x)

        return out[:, :, :res.shape[2]] + res

In [17]:
class TCN(nn.Module):
    def __init__(self, input_dim, output_dim):

        super().__init__()

        self.tcn = nn.Sequential(
            TemporalBlock(input_dim, 32, kernel_size=3, dilation=1),
            TemporalBlock(32, 64, kernel_size=3, dilation=2),
            TemporalBlock(64, 64, kernel_size=3, dilation=4)
        )

        self.fc = nn.Linear(64, output_dim)

    def forward(self, x):

        x = x.transpose(1, 2)

        y = self.tcn(x)

        y = y[:, :, -1]

        return self.fc(y)

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TCN(input_dim=8, output_dim=2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

/home/jacobhardy/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [19]:
for epoch in range(40):

    model.train()
    total_loss = 0

    for X_batch, Y_batch in train_loader:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(pred, Y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch, "Loss:", total_loss / len(train_loader))

Epoch: 0 Loss: 0.0009594631026457336
Epoch: 1 Loss: 0.00013491251290500517
Epoch: 2 Loss: 0.0001222077269524754
Epoch: 3 Loss: 0.000113936585152605
Epoch: 4 Loss: 0.00010689656291816411
Epoch: 5 Loss: 0.00010981283043438983
Epoch: 6 Loss: 0.00010589458578957472
Epoch: 7 Loss: 0.00010828176490025872
Epoch: 8 Loss: 0.00010033860654688278
Epoch: 9 Loss: 0.0001472367419503089
Epoch: 10 Loss: 9.493369442726448e-05
Epoch: 11 Loss: 9.192609645884181e-05
Epoch: 12 Loss: 8.737059811641714e-05
Epoch: 13 Loss: 0.00015157719359978103
Epoch: 14 Loss: 0.00015689321292212052
Epoch: 15 Loss: 8.379667433075196e-05
Epoch: 16 Loss: 8.607894605565025e-05
Epoch: 17 Loss: 8.27481326534299e-05
Epoch: 18 Loss: 0.00011049953981501944
Epoch: 19 Loss: 8.393376385462751e-05
Epoch: 20 Loss: 8.291817252640188e-05
Epoch: 21 Loss: 8.65223987967929e-05
Epoch: 22 Loss: 8.510843200668996e-05
Epoch: 23 Loss: 8.212966590657291e-05
Epoch: 24 Loss: 8.68628118625871e-05
Epoch: 25 Loss: 8.63943882473493e-05
Epoch: 26 Loss: 8.

In [20]:
model.eval()

preds = []
targets = []
inputs = []

with torch.no_grad():

    for X_batch, Y_batch in test_loader:

        X_batch = X_batch.to(device)

        pred = model(X_batch).cpu()

        preds.append(pred)
        targets.append(Y_batch)
        inputs.append(X_batch.cpu())

preds = torch.cat(preds).numpy()
targets = torch.cat(targets).numpy()
inputs = torch.cat(inputs).numpy()

In [21]:
last_lat = inputs[:, -1, 0]
last_lon = inputs[:, -1, 1]

In [22]:
lat_pred = last_lat + preds[:,0]
lon_pred = last_lon + preds[:,1]

lat_true = last_lat + targets[:,0]
lon_true = last_lon + targets[:,1]

In [23]:
import numpy as np

lat_error = lat_pred - lat_true
lon_error = lon_pred - lon_true

rmse = np.sqrt(np.mean(lat_error**2 + lon_error**2))
mae = np.mean(np.sqrt(lat_error**2 + lon_error**2))

print("RMSE:", rmse)
print("MAE:", mae)

RMSE: 0.010658018
MAE: 0.0072067245


In [24]:
R = 6371000

lat_error_rad = np.radians(lat_error)
lon_error_rad = np.radians(lon_error)

lat_true_rad = np.radians(lat_true)

x = lon_error_rad * np.cos(lat_true_rad)
y = lat_error_rad

dist_m = R * np.sqrt(x**2 + y**2)

rmse_m = np.sqrt(np.mean(dist_m**2))
mae_m = np.mean(dist_m)

print("RMSE (meters):", rmse_m)
print("MAE (meters):", mae_m)

RMSE (meters): 1184.927
MAE (meters): 801.26025
